# Discount Calendar Optimization

This notebook creates an optimized discount calendar to maximize revenue.


In [ ]:
import sys
import os
sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'src'))

import pandas as pd
import numpy as np
from utils import load_data, save_data
from model_xgboost import train_xgboost
from discount_calendar import create_season_product_discount_calendar, create_discount_calendar_heatmap

# Load cleaned grocery data
data_path = '../data/cleaned/grocery_cleaned.csv'
df_groceries = load_data(data_path)

# Load features
features_path = '../data/processed/grocery_features.csv'
df = load_data(features_path)

# Prepare features and target (using 'Demand' as per report)
target_col = 'Demand' if 'Demand' in df.columns else 'sales'
feature_cols = [col for col in df.columns if col not in [target_col, 'date', 'Date']]
X = df[feature_cols]
y = df[target_col]

print(f"Training XGBoost model on {len(X)} samples with {len(feature_cols)} features...")
# Train model for forecasting
xgb_model, _, _, _, _ = train_xgboost(X, y)
print("Model trained successfully!")


In [ ]:
# Prepare grocery data with season mapping
# Map months to seasons
season_map = {
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Fall', 10: 'Fall', 11: 'Fall'
}

# Ensure we have Month column
if 'Month' not in df_groceries.columns and 'date' in df_groceries.columns:
    df_groceries['date'] = pd.to_datetime(df_groceries['date'])
    df_groceries['Month'] = df_groceries['date'].dt.month

df_groceries['Season'] = df_groceries['Month'].map(season_map)

print(f"Data prepared: {len(df_groceries)} rows")
print(f"Products: {df_groceries['Product ID'].nunique() if 'Product ID' in df_groceries.columns else 'N/A'}")
print(f"Seasons: {df_groceries['Season'].unique()}")


In [ ]:
# Create Season × Product discount calendar
# This matches the methodology from the report:
# For each Product × Season combination, test discount scenarios (0%, 5%, 10%, ..., 25%)
# Predict demand for each scenario and calculate revenue
# Select discount that maximizes revenue

print("Creating Season × Product discount calendar...")
print("This may take a few minutes...")

discount_calendar = create_season_product_discount_calendar(
    df_groceries,
    xgb_model,
    feature_cols,
    product_col='Product ID',
    season_col='Season',
    discount_col='Discount',
    price_col='Price',
    demand_col='Demand' if 'Demand' in df_groceries.columns else 'sales'
)

print(f"\nDiscount calendar created: {len(discount_calendar)} Product × Season combinations")
print("\nSample of discount calendar:")
print(discount_calendar.head(10))


In [ ]:
# Create and save heatmap visualization
print("\nCreating discount calendar heatmap...")
fig = create_discount_calendar_heatmap(
    discount_calendar,
    product_col='Product ID',
    season_col='Season',
    discount_col='Optimal_Discount',
    save_path='../reports/figures/discount_calendar.png'
)
print("Heatmap saved to ../reports/figures/discount_calendar.png")

# Save discount calendar CSV
output_path = '../data/processed/discount_calendar.csv'
save_data(discount_calendar, output_path)
print(f"\nDiscount calendar saved to {output_path}")

# Display summary statistics
print("\n" + "="*60)
print("DISCOUNT CALENDAR SUMMARY")
print("="*60)
print(f"Total Product × Season combinations: {len(discount_calendar)}")
print(f"Average optimal discount: {discount_calendar['Optimal_Discount'].mean():.2f}%")
print(f"Discount range: {discount_calendar['Optimal_Discount'].min():.0f}% - {discount_calendar['Optimal_Discount'].max():.0f}%")
print(f"Total estimated revenue: ${discount_calendar['Estimated_Revenue'].sum():,.2f}")
